In [ ]:
# %%
import gradio as gr
import os
import json
import time
import random
from datetime import datetime
from pathlib import Path

In [ ]:
# --- CRITICAL CONFIGURATION CHANGES ---
# Assuming the clean manifest is in the same directory as the image folders' parent
MANIFEST_PATH = "./ratings_interface/rlhf_generation_data/generation_manifest.jsonl"
# Point to the directory containing the prompt hash subfolders (real data)
IMAGE_DIR = "./ratings_interface/rlhf_generation_data" 
# Use a new rating file name to separate from mock ratings
RATINGS_FILE = "./ratings_interface/ratings_real.jsonl"
# Ensure the ratings interface directory exists for saving the results
Path("ratings_interface").mkdir(exist_ok=True)

# New Global State: Maps prompt_hash (ID) to the full prompt string
# We only need to load one of the A/B samples since the prompt is the same
PROMPT_MAP = {}
if os.path.exists(MANIFEST_PATH):
    print(f"Loading prompts from {MANIFEST_PATH}...")
    with open(MANIFEST_PATH, 'r') as f:
        for line in f:
            try:
                data = json.loads(line)
                PROMPT_MAP[data['prompt_hash']] = data['prompt']
            except json.JSONDecodeError:
                pass
print(f"Loaded {len(PROMPT_MAP)} unique prompts.")
# -------------------------------------

In [ ]:
# --- 1. Helper Functions (MODIFIED) ---

def find_image_pairs(directory):
    """Scans subdirectories (prompt hashes) for real A/B image pairs."""
    pairs = {}
    
    if not os.path.isdir(directory):
        print(f"❌ Error: Image directory not found at {directory}. Check your paths.")
        return []

    # 1. Iterate through each subdirectory (which is the prompt hash)
    for prompt_hash in os.listdir(directory):
        prompt_hash_path = os.path.join(directory, prompt_hash)
        
        # Skip files and ensure the name is a 10-char hash directory
        # (This avoids reading the generation_manifest.jsonl file)
        if not os.path.isdir(prompt_hash_path) or len(prompt_hash) != 10:
            continue
            
        # Define the expected paths for the A and B images
        path_a = os.path.join(prompt_hash_path, f"{prompt_hash}_A.png")
        path_b = os.path.join(prompt_hash_path, f"{prompt_hash}_B.png")

        # 2. Check if both files exist
        if os.path.exists(path_a) and os.path.exists(path_b):
            pairs[prompt_hash] = {
                "A": path_a,
                "B": path_b
            }
            
    # Format and return, sorted by hash ID
    pair_list = [{"id": prefix, **paths} for prefix, paths in pairs.items()]
    
    # Shuffle the list for unbiased rating order
    random.shuffle(pair_list)
    
    return pair_list

def load_rated_ids(ratings_file):
    """Loads all prompt_ids from the ratings file to avoid re-rating."""
    rated_ids = set()
    if os.path.exists(ratings_file):
        with open(ratings_file, 'r') as f:
            for line in f:
                try:
                    data = json.loads(line)
                    # The 'prompt_id' is the 10-character hash
                    if 'prompt_id' in data:
                        rated_ids.add(data['prompt_id'])
                except (json.JSONDecodeError, KeyError):
                    print(f"Warning: Skipping corrupted line in ratings file: {line}")
    return rated_ids


In [ ]:
# --- 2. Data Loading and Filtering ---

all_pairs = find_image_pairs(IMAGE_DIR)
rated_ids = load_rated_ids(RATINGS_FILE)

# Filter out the pairs that have already been rated
unrated_pairs = [p for p in all_pairs if p['id'] not in rated_ids]

print(f"Found {len(all_pairs)} total pairs in the directory.")
print(f"Found {len(rated_ids)} already rated pairs in '{RATINGS_FILE}'.")
print(f"🚀 Starting rating session with {len(unrated_pairs)} pairs remaining.")


In [ ]:
# --- 3. Gradio App Logic (Unchanged) ---

def save_preference(pair_id, preferred_path, rejected_path):
    """Appends a PREFERRED rating to the JSON Lines file."""
    rating = {
        "prompt_id": pair_id,
        "preferred_img": os.path.basename(preferred_path),
        "rejected_img": os.path.basename(rejected_path),
        "rating_status": "PREFERRED",  # Flag for successful preference
        "timestamp": datetime.utcnow().isoformat()
    }
    with open(RATINGS_FILE, 'a') as f:
        f.write(json.dumps(rating) + '\n')
    print(f"Saved rating for ID {pair_id}: '{os.path.basename(preferred_path)}' was preferred.")

def save_skip(pair_id, path_a, path_b):
    """Appends a SKIPPED rating to the JSON Lines file."""
    rating = {
        "prompt_id": pair_id,
        # Store both paths as skipped for later analysis
        "rejected_img_A": os.path.basename(path_a),
        "rejected_img_B": os.path.basename(path_b),
        "rating_status": "SKIPPED_BAD_FIT", # Flag for skipped pair
        "timestamp": datetime.utcnow().isoformat()
    }
    with open(RATINGS_FILE, 'a') as f:
        f.write(json.dumps(rating) + '\n')
    print(f"Skipped pair ID {pair_id}. Both images flagged as bad fit.")

def get_next_pair(current_index):
    """Loads the next pair of images or ends the session if complete."""
    current_index += 1
    
    # Define a default prompt for error/completion state
    current_prompt = "" 
    
    if current_index >= len(unrated_pairs):
        # All images have been rated (10 items returned)
        completion_message = "✅ All pairs rated! Session complete. You can close this tab."
        print("\n" + completion_message)
        time.sleep(2)
        
        # Returns 10 items: 2x None (images), 1x message, 3x button updates, 1x index, 2x empty paths (states), 1x empty prompt
        return None, None, completion_message, \
               gr.update(interactive=False), gr.update(interactive=False), gr.update(interactive=False), \
               current_index, "", "", current_prompt # <-- ADDED current_prompt

    pair = unrated_pairs[current_index]
    
    # --- NEW: Get Prompt ---
    pair_id = pair['id']
    current_prompt = PROMPT_MAP.get(pair_id, f"Prompt ID: {pair_id}") # Fallback if not found
    # -----------------------
    
    progress = f"Rating pair: {pair_id} ({current_index + 1} of {len(unrated_pairs)})"
    
    # Randomly swap A and B for presentation to prevent left-bias
    if random.random() < 0.5:
        path_left = pair["B"]
        path_right = pair["A"]
    else:
        path_left = pair["A"]
        path_right = pair["B"]
        
    # All returns must now include the 10 items
    return (
        path_left, # img_a
        path_right, # img_b
        progress, # info_box
        gr.update(interactive=True), # left_button
        gr.update(interactive=True), # right_button
        gr.update(interactive=True), # skip_button
        current_index, # current_index
        path_left, # path_a_displayed (STATE)
        path_right, # path_b_displayed (STATE)
        current_prompt # <-- ADDED current_prompt
    )

def choose_preference(current_index, choice, path_a_displayed, path_b_displayed):
    """Handler for choosing left or right (A or B), correcting for the random display swap."""
    if 0 <= current_index < len(unrated_pairs):
        # Gradio passes the paths currently displayed on the screen
        if choice == "A": # User chose the image on the LEFT
            preferred_path = path_a_displayed
            rejected_path = path_b_displayed
        else: # Choice is "B" (User chose the image on the RIGHT)
            preferred_path = path_b_displayed
            rejected_path = path_a_displayed
            
        pair = unrated_pairs[current_index]
        save_preference(pair['id'], preferred_path, rejected_path)
        
    # Load the next pair regardless of choice
    return get_next_pair(current_index)

def choose_skip(current_index):
    """Handler for choosing to skip the pair."""
    if 0 <= current_index < len(unrated_pairs):
        pair = unrated_pairs[current_index]
        save_skip(pair['id'], pair['A'], pair['B'])
    # Load the next pair
    return get_next_pair(current_index)


In [ ]:
# --- 4. UI Definition (UPDATED FOR MOBILE LAYOUT) ---
with gr.Blocks() as demo:
    # State components
    current_index = gr.State(value=-1)
    path_a_displayed = gr.State(value="")
    path_b_displayed = gr.State(value="")

    gr.Markdown("# Image Comparison for RLHF Rating")

    gr.Markdown("Click which image is better, or **Skip / Bad Pair** if neither is a good match for the prompt. **NOTE**: Images are randomly swapped left/right to prevent bias.")
    
    info_box = gr.Markdown("Loading first pair...")

    # NEW: Display the Prompt prominently
    prompt_box = gr.Markdown("## Loading Prompt...") 
        
    with gr.Row(equal_height=True):
        img_a = gr.Image(label="Image A", type="filepath")
        img_b = gr.Image(label="Image B", type="filepath")

    # --- BUTTONS ROW 1: Left and Right (Side-by-side) ---
    with gr.Row():
        # Note: We keep scale=1 to ensure they fill the available width evenly
        left_button = gr.Button("⬅️ Left is Better", variant="primary", scale=1)
        right_button = gr.Button("Right is Better ➡️", variant="primary", scale=1)

    # --- BUTTONS ROW 2: Skip (Centered) ---
    # Using a Row to ensure it takes full width, which centers the button on wide screens, 
    # but the scale=1 means it will still use the full available width of the row.
    with gr.Row():
        skip_button = gr.Button("Skip / Bad Pair", scale=1)

    # --- 5. Event Listeners (UNMODIFIED FROM PREVIOUS STEP) ---

    # Define the NEW FULL list of output components (10 items)
    OUTPUTS_LIST = [
        img_a, img_b, info_box, 
        left_button, right_button, skip_button, 
        current_index, path_a_displayed, path_b_displayed,
        prompt_box
    ]

    # Custom function to update the path states before calling choose_preference
    def update_paths_and_choose(idx, choice, path_a, path_b):
        # This function handles the rating logic and then calls get_next_pair()
        return choose_preference(idx, choice, path_a, path_b)
        
    # All event handlers use the new 10-item OUTPUTS_LIST
    
    # Left Button Click Event
    left_button.click(
        fn=lambda idx, p_a, p_b: update_paths_and_choose(idx, "A", p_a, p_b),
        inputs=[current_index, path_a_displayed, path_b_displayed],
        outputs=OUTPUTS_LIST
    )

    # Right Button Click Event
    right_button.click(
        fn=lambda idx, p_a, p_b: update_paths_and_choose(idx, "B", p_a, p_b),
        inputs=[current_index, path_a_displayed, path_b_displayed],
        outputs=OUTPUTS_LIST
    )

    # Skip Button Click Event
    skip_button.click(
        fn=choose_skip,
        inputs=[current_index],
        outputs=OUTPUTS_LIST
    )

    # Initial Load Event: Calls get_next_pair directly.
    demo.load(
        fn=get_next_pair,
        inputs=[current_index], # Passes -1 (initial state)
        outputs=OUTPUTS_LIST
    )

# --- 6. Launch the App ---
if len(unrated_pairs) == 0:
    print("✅ No unrated images found. Nothing to do.")
else:
    demo.launch(share=False, quiet=True)